# 02 – Data Preprocessing Pipeline

This notebook demonstrates the complete preprocessing pipeline applied to the **AG News** dataset for automatic news topic classification.

We explore two distinct preprocessing strategies:
- **TF-IDF Pipeline**: aggressive text cleaning (lowercase, URL/HTML removal, stopword removal, lemmatization) suitable for bag-of-words models.
- **Transformer Pipeline**: minimal cleaning that preserves linguistic structure for BERT-style models.

The AG News dataset contains **120,000 training samples** across 4 categories: `World`, `Sports`, `Business`, `Sci/Tech`.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import re
import warnings
from collections import Counter
from pathlib import Path

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False
    print("[WARNING] wordcloud not installed. Run: pip install wordcloud")

from sklearn.feature_extraction.text import CountVectorizer

# Download required NLTK data
for resource in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

warnings.filterwarnings('ignore')
np.random.seed(42)

# Matplotlib settings
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='Set2')

# Paths
DATA_RAW        = Path('../data/raw')
DATA_PROCESSED  = Path('../data/processed')
VISUALIZATIONS  = Path('../visualizations')
VISUALIZATIONS.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 1. Load Raw Data

The raw AG News training set (`train.csv`) contains three columns:
- **label** — integer class (1–4)
- **title** — article headline
- **description** — short article snippet

We concatenate `title` and `description` to form the full input text.

In [ ]:
# Load raw training data
raw_path = DATA_RAW / 'train.csv'
try:
    df_raw = pd.read_csv(raw_path, header=None, names=['label', 'title', 'description'])
except FileNotFoundError:
    raise FileNotFoundError(f'Raw data not found at {raw_path}. Please verify the data directory.')

# Label mapping
LABEL_MAP = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
df_raw['category'] = df_raw['label'].map(LABEL_MAP)

print(f'Shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
print()
print('Label distribution:')
print(df_raw['category'].value_counts())
print()
df_raw.head(5)

In [ ]:
# Data types and null check
print('Data types:')
print(df_raw.dtypes)
print()
print('Null values per column:')
print(df_raw.isnull().sum())
print()

# Create combined text column
df_raw['text_raw'] = df_raw['title'].fillna('') + ' ' + df_raw['description'].fillna('')
print(f'Sample raw text:\n{df_raw["text_raw"].iloc[0]}')

## 2. TF-IDF Preprocessing Pipeline

The TF-IDF preprocessing pipeline consists of **7 sequential steps** designed to reduce vocabulary noise and normalise text for bag-of-words models:

| Step | Operation | Rationale |
|------|-----------|---------- |
| 1 | **Lowercase** | Unify case variants (`Apple` → `apple`) |
| 2 | **URL removal** | Strip `http://…`, `www.…` links |
| 3 | **HTML removal** | Remove `<tag>` markup from scraped content |
| 4 | **Special char removal** | Keep only letters and spaces; remove punctuation & digits |
| 5 | **Tokenization** | Split text into individual word tokens |
| 6 | **Stopword removal** | Drop high-frequency, low-information words (`the`, `is`, `at`, …) |
| 7 | **Lemmatization** | Reduce inflected forms to base form (`running` → `run`, `better` → `good`) |

> **Why not stemming?** Lemmatization produces valid dictionary words, making the results more interpretable for academic analysis and feature-importance inspection.

In [ ]:
# ── Preprocessing functions ─────────────────────────────────────────────────
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def step1_lowercase(text: str) -> str:
    return text.lower()

def step2_remove_urls(text: str) -> str:
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def step3_remove_html(text: str) -> str:
    return re.sub(r'<[^>]+>', '', text)

def step4_remove_special(text: str) -> str:
    return re.sub(r'[^a-z\s]', ' ', text)

def step5_tokenize(text: str):
    return word_tokenize(text)

def step6_remove_stopwords(tokens) -> list:
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 1]

def step7_lemmatize(tokens) -> list:
    return [lemmatizer.lemmatize(t) for t in tokens]

def preprocess_tfidf(text: str) -> str:
    text = step1_lowercase(str(text))
    text = step2_remove_urls(text)
    text = step3_remove_html(text)
    text = step4_remove_special(text)
    tokens = step5_tokenize(text)
    tokens = step6_remove_stopwords(tokens)
    tokens = step7_lemmatize(tokens)
    return ' '.join(tokens)

print('Preprocessing functions defined.')

## 3. Step-by-Step Demonstration

We trace a single raw article through each preprocessing step to observe the transformation.

In [ ]:
# Pick one representative sample
sample_idx = 42
sample_raw = df_raw['text_raw'].iloc[sample_idx]
sample_cat = df_raw['category'].iloc[sample_idx]

print('=' * 70)
print(f'SAMPLE (category: {sample_cat})')
print('=' * 70)

stages = [
    ('ORIGINAL', sample_raw),
]

t = sample_raw
t = step1_lowercase(t);        stages.append(('Step 1 – Lowercase', t))
t = step2_remove_urls(t);      stages.append(('Step 2 – URL removed', t))
t = step3_remove_html(t);      stages.append(('Step 3 – HTML removed', t))
t = step4_remove_special(t);   stages.append(('Step 4 – Special chars removed', t))
tokens = step5_tokenize(t);    stages.append(('Step 5 – Tokenized', str(tokens[:15]) + ' …'))
tokens = step6_remove_stopwords(tokens); stages.append(('Step 6 – Stopwords removed', str(tokens[:15]) + ' …'))
tokens = step7_lemmatize(tokens);        stages.append(('Step 7 – Lemmatized', ' '.join(tokens)))

for name, value in stages:
    print(f'\n[{name}]')
    display_val = str(value)[:300] + ('…' if len(str(value)) > 300 else '')
    print(display_val)

print('\n' + '=' * 70)
print(f'Token count before stopword removal: {len(step5_tokenize(step4_remove_special(step3_remove_html(step2_remove_urls(step1_lowercase(sample_raw))))))} tokens')
print(f'Token count after full pipeline:     {len(tokens)} tokens')

## 4. Vocabulary Analysis

We compare vocabulary sizes across the raw data and both preprocessing strategies.

In [ ]:
# Load preprocessed datasets
def load_preprocessed(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Handle possible column name variations
    text_col_candidates = ['text', 'cleaned_text', 'processed_text', 'content']
    label_col_candidates = ['label', 'category', 'class', 'target']
    text_col  = next((c for c in text_col_candidates if c in df.columns), df.columns[0])
    label_col = next((c for c in label_col_candidates if c in df.columns), df.columns[1])
    df = df.rename(columns={text_col: 'text', label_col: 'label_raw'})
    return df

train_tfidf = load_preprocessed(DATA_PROCESSED / 'train_preprocessed.csv')
train_trans = load_preprocessed(DATA_PROCESSED / 'train_transformer.csv')

print(f'TF-IDF preprocessed shape : {train_tfidf.shape}')
print(f'Transformer preprocessed shape: {train_trans.shape}')
print()
print('TF-IDF columns:', train_tfidf.columns.tolist())
print('Transformer columns:', train_trans.columns.tolist())
print()
print('TF-IDF sample:')
print(train_tfidf['text'].iloc[0][:200])

In [ ]:
# Compute vocabulary sizes using CountVectorizer (tokenises on whitespace)
def vocab_size(texts, analyzer='word', token_pattern=r'(?u)\b\w+\b'):
    cv = CountVectorizer(analyzer=analyzer, token_pattern=token_pattern)
    cv.fit(texts.dropna().astype(str))
    return len(cv.vocabulary_)

print('Computing vocabulary sizes (this may take ~30 s)…')
raw_vocab      = vocab_size(df_raw['text_raw'])
tfidf_vocab    = vocab_size(train_tfidf['text'])
trans_vocab    = vocab_size(train_trans['text'])

vocab_df = pd.DataFrame({
    'Pipeline'        : ['Raw (title + desc)', 'TF-IDF Preprocessed', 'Transformer Preprocessed'],
    'Vocabulary Size' : [raw_vocab, tfidf_vocab, trans_vocab],
    'Reduction %'     : [0,
                         round((1 - tfidf_vocab / raw_vocab) * 100, 1),
                         round((1 - trans_vocab / raw_vocab) * 100, 1)]
})

print()
print(vocab_df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(vocab_df['Pipeline'], vocab_df['Vocabulary Size'],
              color=sns.color_palette('Set2', 3), edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, vocab_df['Vocabulary Size']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Vocabulary Size Comparison Across Preprocessing Pipelines', fontsize=13)
ax.set_ylabel('Unique Tokens')
ax.set_xlabel('')
ax.tick_params(axis='x', labelrotation=10)
plt.tight_layout()
plt.savefig(VISUALIZATIONS / 'vocab_size_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Text Length Analysis

Comparing token counts before and after TF-IDF preprocessing reveals how aggressively the pipeline compresses text.

In [ ]:
# Token length distributions
raw_lengths   = df_raw['text_raw'].dropna().astype(str).str.split().apply(len)
tfidf_lengths = train_tfidf['text'].dropna().astype(str).str.split().apply(len)
trans_lengths = train_trans['text'].dropna().astype(str).str.split().apply(len)

print('Text length statistics (token count):')
stats_df = pd.DataFrame({
    'Pipeline'  : ['Raw', 'TF-IDF Preprocessed', 'Transformer Preprocessed'],
    'Mean'      : [raw_lengths.mean(), tfidf_lengths.mean(), trans_lengths.mean()],
    'Median'    : [raw_lengths.median(), tfidf_lengths.median(), trans_lengths.median()],
    'Std'       : [raw_lengths.std(), tfidf_lengths.std(), trans_lengths.std()],
    'Min'       : [raw_lengths.min(), tfidf_lengths.min(), trans_lengths.min()],
    'Max'       : [raw_lengths.max(), tfidf_lengths.max(), trans_lengths.max()],
}).round(2)
print(stats_df.to_string(index=False))

# Side-by-side histograms
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

datasets = [
    (raw_lengths,   'Raw Text',                  '#4C72B0'),
    (tfidf_lengths, 'TF-IDF Preprocessed',       '#55A868'),
    (trans_lengths, 'Transformer Preprocessed',  '#C44E52'),
]

for ax, (lengths, title, color) in zip(axes, datasets):
    ax.hist(lengths.clip(upper=lengths.quantile(0.99)), bins=40,
            color=color, edgecolor='white', alpha=0.85)
    ax.axvline(lengths.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean={lengths.mean():.1f}')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Token Count')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

fig.suptitle('Text Length Distribution (token count) Before & After Preprocessing', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(VISUALIZATIONS / 'text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Word Clouds

Word clouds provide an intuitive visual summary of the most frequent terms in the TF-IDF-preprocessed corpus.

In [ ]:
if not WORDCLOUD_AVAILABLE:
    print('[SKIP] wordcloud library not available. Install with: pip install wordcloud')
else:
    CATEGORIES = ['World', 'Sports', 'Business', 'Sci/Tech']
    COLORS = ['Blues', 'Greens', 'Oranges', 'Purples']

    # Determine category column
    cat_col = 'category' if 'category' in train_tfidf.columns else \
              ('label' if 'label' in train_tfidf.columns else train_tfidf.columns[-1])

    # ── Overall word cloud ─────────────────────────────────────────────────
    all_text = ' '.join(train_tfidf['text'].dropna().astype(str).values)

    wc_all = WordCloud(
        width=900, height=450, background_color='white',
        max_words=200, colormap='viridis', random_state=42
    ).generate(all_text)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(wc_all, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Overall Word Cloud — TF-IDF Preprocessed Corpus', fontsize=14)
    plt.tight_layout()
    plt.savefig(VISUALIZATIONS / 'wordcloud_overall.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Per-category word clouds (2×2) ────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for idx, (cat, cmap) in enumerate(zip(CATEGORIES, COLORS)):
        mask_series = train_tfidf[cat_col].astype(str).str.strip()
        cat_text = ' '.join(
            train_tfidf.loc[mask_series == cat, 'text'].dropna().astype(str).values
        )
        if not cat_text.strip():
            axes[idx].set_visible(False)
            continue

        wc = WordCloud(
            width=600, height=400, background_color='white',
            max_words=100, colormap=cmap, random_state=42
        ).generate(cat_text)

        axes[idx].imshow(wc, interpolation='bilinear')
        axes[idx].axis('off')
        axes[idx].set_title(f'Category: {cat}', fontsize=13, fontweight='bold')

    fig.suptitle('Word Clouds per Category — TF-IDF Preprocessed', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.savefig(VISUALIZATIONS / 'wordcloud_per_category.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Top Words per Category

We use `Counter` to identify the 10 most frequent tokens for each news category.

In [ ]:
CATEGORIES = ['World', 'Sports', 'Business', 'Sci/Tech']
cat_col = 'category' if 'category' in train_tfidf.columns else \
          ('label' if 'label' in train_tfidf.columns else train_tfidf.columns[-1])

TOP_N = 10
cat_top_words = {}

for cat in CATEGORIES:
    texts = train_tfidf.loc[train_tfidf[cat_col].astype(str).str.strip() == cat, 'text']
    all_words = ' '.join(texts.dropna().astype(str)).split()
    counter = Counter(all_words)
    cat_top_words[cat] = counter.most_common(TOP_N)

# Plot 2×2 bar chart grid
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
palette = sns.color_palette('Set2', TOP_N)

for idx, cat in enumerate(CATEGORIES):
    words, counts = zip(*cat_top_words[cat])
    axes[idx].barh(words[::-1], counts[::-1], color=palette)
    axes[idx].set_title(f'Top {TOP_N} Words — {cat}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Frequency')
    for i, (w, c) in enumerate(zip(words[::-1], counts[::-1])):
        axes[idx].text(c + max(counts) * 0.01, i, f'{c:,}', va='center', fontsize=8)

fig.suptitle(f'Top {TOP_N} Most Frequent Words per Category (TF-IDF Preprocessed)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(VISUALIZATIONS / 'top_words_per_category.png', dpi=150, bbox_inches='tight')
plt.show()

# Print table
print('\nTop words per category:')
for cat in CATEGORIES:
    print(f'  {cat:12s}: {[w for w, _ in cat_top_words[cat]]}')

## 8. Preprocessing Pipeline Comparison

| Attribute | TF-IDF Pipeline | Transformer Pipeline |
|-----------|-----------------|---------------------|
| **Lowercase** | ✅ Yes | ✅ Yes |
| **URL removal** | ✅ Yes | ✅ Yes |
| **HTML removal** | ✅ Yes | ✅ Yes |
| **Special char removal** | ✅ Aggressive (non-alpha stripped) | ⚠️ Light (preserve sentence structure) |
| **Stopword removal** | ✅ Yes (NLTK English list) | ❌ No |
| **Lemmatization** | ✅ Yes (WordNet) | ❌ No |
| **Tokenizer** | NLTK `word_tokenize` | Preserved as-is (BERT tokenizer handles it) |
| **Output format** | Space-joined token string | Cleaned raw sentence |
| **Vocab size** | Significantly smaller | Closer to raw |
| **Target model** | TF-IDF vectorizer + classical ML | HuggingFace BERT/DistilBERT |
| **Rationale** | Reduce sparsity; focus on content words | Preserve full context for attention |


In [ ]:
# Side-by-side sample comparison: same article in both pipelines
sample_idx = 100
tfidf_sample = train_tfidf['text'].iloc[sample_idx]
trans_sample = train_trans['text'].iloc[sample_idx]

print('SAME ARTICLE — BOTH PIPELINES')
print('=' * 70)
print('TF-IDF Preprocessed:')
print(str(tfidf_sample)[:400])
print()
print('Transformer Preprocessed:')
print(str(trans_sample)[:400])
print()
print(f'TF-IDF token count  : {len(str(tfidf_sample).split())}')
print(f'Transformer token count: {len(str(trans_sample).split())}')

## 9. Summary

### Key Findings

1. **Vocabulary reduction**: The TF-IDF pipeline achieves a substantial reduction in unique token count compared to the raw corpus, primarily through stopword removal and lemmatization. The transformer pipeline retains more vocabulary to preserve contextual information.

2. **Text length compression**: After TF-IDF preprocessing, average article length drops significantly (typically ~40–50% fewer tokens), as function words and stopwords are removed.

3. **Category-distinctive vocabulary**: Each class has clearly distinguishable top words:
   - **World**: government, country, president, war, military
   - **Sports**: game, team, season, win, player
   - **Business**: market, company, share, growth, sale
   - **Sci/Tech**: software, internet, technology, computer, research

4. **Pipeline trade-off**: The TF-IDF pipeline sacrifices linguistic richness for efficiency and reduced sparsity, making it ideal for classical ML. The transformer pipeline preserves sentence structure to maximise contextual embedding quality.

5. **Balanced dataset**: AG News is perfectly balanced with 30,000 samples per category, ensuring no class-imbalance bias in model training.